# Preentrenamiento del reconocedor de LSE en GPU

Este cuaderno preentrena el modelo grande con las grabaciones de lengua de signos americana,
usando la GPU gratuita de Colab. Lo que en un procesador tarda muchas horas, aqui son minutos.

## Antes de empezar
1. En Colab, menu **Entorno de ejecucion -> Cambiar tipo de entorno -> GPU**.
2. Sube a tu Google Drive una carpeta llamada **lse_colab** con estos cuatro archivos:
   X_asl_train.npy, X_asl_val.npy, y_asl_train.npy, y_asl_val.npy
3. Ejecuta las celdas en orden.

Al terminar, el modelo entrenado queda en tu Drive como preentrenado_asl_arq128_gpu.keras.
Descargalo y pasamelo para integrarlo en la aplicacion.

In [ ]:
# 1. Comprobar GPU y conectar Google Drive
import tensorflow as tf
print("GPU disponible:", tf.config.list_physical_devices('GPU'))
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Definiciones: preparacion de datos, modelo y entrenamiento
import numpy as np, math
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def anadir_velocidad(X):
    presente = np.any(X != 0, axis=-1)
    velocidad = np.zeros_like(X)
    velocidad[:, 1:] = X[:, 1:] - X[:, :-1]
    valido = presente[:, 1:] & presente[:, :-1]
    velocidad[:, 1:][~valido] = 0
    return np.concatenate([X, velocidad], axis=-1)

def a_entrada_modelo(X):
    X = anadir_velocidad(X)
    n, f, p, c = X.shape
    return X.reshape(n, f, p * c)

class Lotes(keras.utils.PyDataset):
    def __init__(self, X, y, tam=32, **kw):
        super().__init__(**kw); self.X=X; self.y=y; self.tam=tam
    def __len__(self): return math.ceil(self.X.shape[0]/self.tam)
    def __getitem__(self, i):
        a=i*self.tam; b=a+self.tam
        lote = a_entrada_modelo(np.asarray(self.X[a:b]).astype("float32"))
        return lote, self.y[a:b]

def construir_transformer(num_pasos, num_rasgos, num_signos, dimension=128, cabezas=4, oculta=128, bloques=3, dropout=0.3):
    entradas = keras.Input(shape=(num_pasos, num_rasgos))
    validos = tf.reduce_any(tf.not_equal(entradas, 0.0), axis=-1)
    mascara = tf.expand_dims(validos, axis=1)
    x = layers.Dense(dimension)(entradas)
    x = x + layers.Embedding(num_pasos, dimension)(tf.range(0, num_pasos))
    for _ in range(bloques):
        a = layers.MultiHeadAttention(num_heads=cabezas, key_dim=dimension//cabezas)(x, x, attention_mask=mascara)
        x = layers.LayerNormalization()(x + a)
        f = layers.Dense(oculta, activation="relu")(x); f = layers.Dense(dimension)(f)
        x = layers.LayerNormalization()(x + f)
    peso = tf.expand_dims(tf.cast(validos, x.dtype), axis=-1)
    x = tf.reduce_sum(x*peso, axis=1) / tf.maximum(tf.reduce_sum(peso, axis=1), 1.0)
    x = layers.Dropout(dropout)(x)
    salidas = layers.Dense(num_signos, activation="softmax")(x)
    return keras.Model(entradas, salidas)

def compilar(modelo, pasos_por_epoca, epocas, lr_pico=1e-3):
    total = max(1, pasos_por_epoca*epocas); cal = max(1, int(0.05*total))
    ritmo = keras.optimizers.schedules.CosineDecay(0.0, decay_steps=max(1,total-cal), warmup_target=lr_pico, warmup_steps=cal)
    modelo.compile(optimizer=keras.optimizers.AdamW(learning_rate=ritmo, weight_decay=1e-4, clipnorm=1.0),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo
print("Definiciones cargadas")

In [ ]:
# 3. Cargar datos y entrenar en GPU
CARPETA = "/content/drive/MyDrive/lse_colab"
Xtr = np.load(f"{CARPETA}/X_asl_train.npy", mmap_mode="r")
ytr = np.load(f"{CARPETA}/y_asl_train.npy")
Xv = np.load(f"{CARPETA}/X_asl_val.npy", mmap_mode="r")
yv = np.load(f"{CARPETA}/y_asl_val.npy")

num_signos = int(max(ytr.max(), yv.max())) + 1
num_pasos = Xtr.shape[1]
num_rasgos = Xtr.shape[2] * Xtr.shape[3] * 2
print(f"{Xtr.shape[0]} grabaciones, {num_signos} signos")

EPOCAS = 150
tr = Lotes(Xtr, ytr); va = Lotes(Xv, yv)
modelo = construir_transformer(num_pasos, num_rasgos, num_signos, dimension=128, bloques=3)
compilar(modelo, len(tr), EPOCAS, lr_pico=1e-3)
parada = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=25, restore_best_weights=True)
modelo.fit(tr, validation_data=va, epochs=EPOCAS, callbacks=[parada], verbose=2)

modelo.save(f"{CARPETA}/preentrenado_asl_arq128_gpu.keras")
print("Guardado en tu Drive: preentrenado_asl_arq128_gpu.keras")